In [ ]:
# ================================================================
#  04_melt_for_training — wide -> long, schema rút gọn 10 cột, sẵn sàng train
#  Input : do pipeline_config.pair_path() quyết định —
#          USE_DECONFOUNDED=True  -> output/dataset_{split}_deconf.parquet  (02b)
#          USE_DECONFOUNDED=False -> *_normalized.parquet, và *_perturbed.parquet
#                                    cho val/test_ood_model nếu file tồn tại
#  Output: dataset_{split}_final_long.parquet — mỗi row = 1 mẫu sẵn sàng train/eval.
#
#  QUAN TRỌNG (đọc trước khi dùng để train): bucket attack_type="none" là bucket
#  cân bằng 1:1 human/AI, dùng để tính AUC/F1/threshold chuẩn. Các bucket
#  attack_type != "none" chỉ dùng để báo cáo "detection rate under attack" so
#  với CÙNG baseline human ở bucket "none" — KHÔNG pool chung "none" với các
#  attack_type khác rồi tính 1 accuracy duy nhất (sẽ lệch label, xem báo cáo
#  bên dưới).
# ================================================================

import sys
from pathlib import Path

import pandas as pd

# ── CELL 1: Config ──────────────────────────────────────────────
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path(".").resolve()

OUTPUT_DIR = BASE_DIR / "output"

# Bản gốc / bản đã cắt artifact: sửa USE_DECONFOUNDED trong pipeline_config.py.
# Dùng chung với 03_diagnostics -> không thể lệch nhau nữa.
sys.path.insert(0, str(BASE_DIR))
import pipeline_config as cfg  # noqa: E402

INPUT_FILES = {sp: cfg.pair_path(OUTPUT_DIR, sp) for sp in cfg.SPLITS}
print(cfg.banner())

ATTACK_COL_MAP = {
    "ai_text_deepwordbug": "deepwordbug",
    "ai_text_textbugger": "textbugger",
    "ai_text_paraphrase": "paraphrase",
}

FINAL_COLS = [
    "sample_id", "paper_id", "text", "label", "ai_degree", "attack_type",
    "generator_family", "split", "word_count", "sample_weight",
]

for split, path in INPUT_FILES.items():
    flag = "✅" if path.exists() else "❌ KHÔNG TÌM THẤY"
    print(f"  {split:<16}: {flag}  {path.name}")


# ── CELL 2: Helpers ────────────────────────────────────────────
def word_count(text):
    return len(str(text).split()) if text is not None and pd.notna(text) else 0


def emit_row(paper_id, sample_id, text, label, ai_degree, attack_type,
             generator_family, split, sample_weight):
    return {
        "sample_id": sample_id,
        "paper_id": paper_id,
        "text": text,
        "label": label,
        "ai_degree": ai_degree,
        "attack_type": attack_type,
        "generator_family": generator_family,
        "split": split,
        "word_count": word_count(text),
        "sample_weight": sample_weight if pd.notna(sample_weight) else 1.0,
    }


# ── CELL 3: Melt từng loại split ───────────────────────────────
def melt_train(df, split_name):
    rows = []

    for _, r in df.iterrows():
        base_id = r["sample_id"]
        pid = r["paper_id"]
        w = r.get("sample_weight", 1.0)
        family = r.get("generator_family")

        rows.append(emit_row(pid, f"{base_id}_human", r["human_text"], 0, 0,
                              "none", None, split_name, w))
        rows.append(emit_row(pid, f"{base_id}_ai", r["ai_text"], 1, r.get("ai_degree", 3),
                              "none", family, split_name, w))

    return pd.DataFrame(rows)


def melt_eval(df, split_name):
    rows = []

    for _, r in df.iterrows():
        base_id = r["sample_id"]
        pid = r["paper_id"]
        w = r.get("sample_weight", 1.0)
        family = r.get("generator_family")
        degree = r.get("ai_degree", 3)

        rows.append(emit_row(pid, f"{base_id}_human", r["human_text"], 0, 0,
                              "none", None, split_name, w))
        rows.append(emit_row(pid, f"{base_id}_ai", r["ai_text"], 1, degree,
                              "none", family, split_name, w))

        for col, attack_name in ATTACK_COL_MAP.items():
            if col not in df.columns:
                continue

            val = r.get(col)

            if pd.notna(val) and str(val).strip():
                rows.append(emit_row(pid, f"{base_id}_{attack_name}", val, 1, degree,
                                      attack_name, family, split_name, w))

        if "ai_text_llm_mixing" in df.columns:
            val = r.get("ai_text_llm_mixing")

            # Bo qua neu khong doi so voi ai_text goc (llm-attack.ipynb tra ve
            # nguyen ban khi doan qua ngan, was_mixed=False) -- tranh trung row.
            if pd.notna(val) and str(val).strip() and val != r["ai_text"]:
                rows.append(emit_row(pid, f"{base_id}_mixing", val, 1, 2,
                                      "mixing", family, split_name, w))

    return pd.DataFrame(rows)


def melt_test_fp(df, split_name="test_fp"):
    rows = []

    for _, r in df.iterrows():
        base_id = r["sample_id"]
        pid = r["paper_id"]
        w = r.get("sample_weight", 1.0)

        has_ai = pd.notna(r.get("ai_text"))
        text = r["ai_text"] if has_ai else r["human_text"]
        degree = int(r["ai_degree"]) if has_ai else 0

        rows.append(emit_row(pid, f"{base_id}_fp", text, 0, degree, "none", None,
                              split_name, w))

    return pd.DataFrame(rows)


# ── CELL 4: Chạy melt cho từng split ────────────────────────────
final_frames = {}

if INPUT_FILES["train"].exists():
    df = pd.read_parquet(INPUT_FILES["train"])
    final_frames["train"] = melt_train(df, "train")

if INPUT_FILES["val"].exists():
    df = pd.read_parquet(INPUT_FILES["val"])
    final_frames["val"] = melt_eval(df, "val")

if INPUT_FILES["test_ood_model"].exists():
    df = pd.read_parquet(INPUT_FILES["test_ood_model"])
    final_frames["test_ood_model"] = melt_eval(df, "test_ood_model")

if INPUT_FILES["test_fp"].exists():
    df = pd.read_parquet(INPUT_FILES["test_fp"])
    final_frames["test_fp"] = melt_test_fp(df)


# ── CELL 5: QA — đếm label theo (split, attack_type), double-check G1 ──
print("\n" + "=" * 60)
print("BẢNG ĐẾM label theo (split, attack_type)")
print("=" * 60)

all_df = pd.concat(final_frames.values(), ignore_index=True) if final_frames else pd.DataFrame()

if len(all_df) > 0:
    summary = (
        all_df.groupby(["split", "attack_type", "label"])
        .size()
        .unstack(fill_value=0)
        .rename(columns={0: "label=0 (human)", 1: "label=1 (AI)"})
    )
    print(summary.to_string())

    print("\n⚠️  Nhắc lại: chỉ tính accuracy/AUC trên bucket attack_type=='none'. "
          "Các bucket khác chỉ dùng để tính detection-rate-under-attack so với "
          "CÙNG baseline human ở bucket 'none' — không pool chung.")

    print("\nG1 double-check (leakage paper_id giữa split ở tầng long format):")
    leak = (
        all_df.groupby("paper_id")["split"].nunique()
    )
    leak_papers = leak[leak > 1]

    if len(leak_papers) == 0:
        print("  ✅ PASS — không paper_id nào lọt >1 split.")
    else:
        print(f"  ❌ FAIL — {len(leak_papers)} paper_id lọt nhiều split:")
        print(leak_papers.head(20).to_string())
else:
    print("Không có dữ liệu nào được melt — kiểm tra lại INPUT_FILES ở trên.")


# ── CELL 6: Ghi output ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("GHI OUTPUT")
print("=" * 60)

for split_name, df in final_frames.items():
    df = df[FINAL_COLS]
    out_path = cfg.long_path(OUTPUT_DIR, split_name)
    df.to_parquet(out_path, index=False)
    print(f"✅ {out_path.name:<40} {len(df):>7,} dòng")

total = sum(len(df) for df in final_frames.values())
print(f"\nTổng cộng: {total:,} dòng (mục tiêu kế hoạch: ~50,000)")
